In [1]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from statsmodels.tsa.stattools import coint, adfuller
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
np.random.seed(42)

print('All imports successful.')

tier1_pairs = [
    ('CDNS', 'SNPS'),
    ('KLAC', 'ODFL'),
    ('KLAC', 'SNPS'),
    ('ODFL', 'SNPS'),
    ('CDNS', 'KLAC'),
]

tier2_pairs = [
    ('FAST', 'MSFT'),
    ('QCOM', 'TSLA'),
    ('PAYX', 'PEP'),
    ('ADP', 'PEP'),
    ('AMGN', 'CTAS'),
    ('CDW', 'CTAS'),
    ('AMGN', 'CDW'),
]

all_pairs = tier1_pairs + tier2_pairs

All imports successful.


In [2]:
# Re-downloading full price history
import yfinance as yf

tickers = list(set([t for pair in all_pairs for t in pair]))
tickers.sort()

print(f'Downloading price data for {len(tickers)} unique tickers...')
print(f'Tickers: {','.join(tickers)}\n')

raw = yf.download(
    tickers,
    start='2014-01-01',
    end='2024-01-01',
    interval='1wk',
    auto_adjust=True,
    progress=True
)

price_data = raw['Close']
price_data = price_data.resample('W-WED').last()
price_data = price_data.dropna(how='all')
price_data = price_data.ffill().dropna(axis=1)

print(f'\nPrice data shape: {price_data.shape}')
print(f'Date range: {price_data.index[0].date()} to {price_data.index[-1].date()}')
print(f'Tickers retained: {len(price_data.columns)}')
print(f'\nAll tickers present: {all(t in price_data.columns for t in tickers)}')

Tickers: ADP,AMGN,CDNS,CDW,CTAS,FAST,KLAC,MSFT,ODFL,PAYX,PEP,QCOM,SNPS,TSLA



[*********************100%***********************]  14 of 14 completed



Price data shape: (522, 14)
Date range: 2014-01-01 to 2023-12-27
Tickers retained: 14

All tickers present: True


In [3]:
# Defining the three periods: Train, Validate, Test
TRAIN_START = '2014-01-01'
TRAIN_END = '2017-12-31'
VAL_START = '2018-01-01'
VAL_END = '2020-12-31'
TEST_START = '2021-01-01'
TEST_END = '2023-12-31'

# Spliting price data into three periods
train_prices = price_data[TRAIN_START:TRAIN_END]
val_prices = price_data[VAL_START:VAL_END]
test_prices = price_data[TEST_START:TEST_END]

# Printing period summaries
print('Three-Period Data Split')
print('='*70)

for name, df, start, end in [
    ('Training', train_prices, TRAIN_START, TRAIN_END),
    ('Validating', val_prices, VAL_START, VAL_END),
    ('Testing', test_prices, TEST_START, TEST_END)
]:
    print(f'\n{name} Period ({start} to {end}):')
    print(f'  Weeks: {len(df)}')
    print(f'  Start: {df.index[0].date()}')
    print(f'  End: {df.index[-1].date()}')
    print(f'  Tickers: {len(df.columns)}')

Three-Period Data Split

Training Period (2014-01-01 to 2017-12-31):
  Weeks: 209
  Start: 2014-01-01
  End: 2017-12-27
  Tickers: 14

Validating Period (2018-01-01 to 2020-12-31):
  Weeks: 157
  Start: 2018-01-03
  End: 2020-12-30
  Tickers: 14

Testing Period (2021-01-01 to 2023-12-31):
  Weeks: 156
  Start: 2021-01-06
  End: 2023-12-27
  Tickers: 14


In [4]:
def estimate_hedge_ratio(ticker1, ticker2, train_prices):
    """
    Estimates the OLS hedge ratio between two stocks using
    only the training period price data.

    This is the most critical methodological constraint in Notebook 4.
    The hedge ratio is frozen after training and applied
    unchanged to the validation and test periods.

    Returns hedge ratio, intercept and R-squared
    """
    combined = pd.concat(
        [train_prices[ticker1], train_prices[ticker2]], axis=1
    ).dropna()
    combined.columns = [ticker1, ticker2]

    X     = add_constant(combined[ticker2])
    model = OLS(combined[ticker1], X).fit()

    hedge_ratio = float(model.params[ticker2])
    intercept   = float(model.params['const'])
    r_squared   = float(model.rsquared)

    return hedge_ratio, intercept, r_squared

# Compute and store hedge ratios for all 12 pairs
print('Estimating hedge ratios on training data only (2014-2017)...')
print('These ratios will be frozen and applied to all three periods. \n')

hedge_ratios = {}

print(f'{'Pair': <15} {'Hedge Ratio':>12} {'Intercept':>12} {'R-Squared':>12}')
print('-'*70)

for t1, t2 in all_pairs:
    pair_name = f'{t1}/{t2}'
    hedge_ratio, intercept, r_squared = estimate_hedge_ratio(
        t1, t2, train_prices
    )
    hedge_ratios[pair_name] = {
        'ticker1':       t1,
        'ticker2':       t2,
        'hedge_ratio':   round(hedge_ratio, 4),
        'intercept':     round(intercept, 4),
        'r_squared':     round(r_squared, 4)
    }
    print(f'{pair_name:<15} {hedge_ratio:>12.4f} '
          f'{intercept:>12.4f} {r_squared:>12.4f} ')

print(f'\nHedge ratios estimated for {len(hedge_ratios)} pairs.')
print('Training Period: 2014-01-01 to 2017-12-31 (209 weeks)')   

Estimating hedge ratios on training data only (2014-2017)...
These ratios will be frozen and applied to all three periods. 

Pair             Hedge Ratio    Intercept    R-Squared
----------------------------------------------------------------------
CDNS/SNPS             0.5298      -4.6507       0.9639 
KLAC/ODFL             0.2496      -0.0025       0.7223 
KLAC/SNPS             0.1060       0.2788       0.8819 
ODFL/SNPS             0.3405       5.6788       0.7856 
CDNS/KLAC             4.5418      -3.2908       0.9022 
FAST/MSFT             0.0349       6.8309       0.2852 
QCOM/TSLA            -0.3643      53.1894       0.0379 
PAYX/PEP              0.6640     -10.7111       0.8777 
ADP/PEP               1.2323     -20.0479       0.8991 
AMGN/CTAS             2.1242      67.8748       0.7181 
CDW/CTAS              1.7830       0.1472       0.9546 
AMGN/CDW              1.1213      70.3926       0.6664 

Hedge ratios estimated for 12 pairs.
Training Period: 2014-01-01 to 2017-12-